In [ ]:
# CELL 1

import os
import glob
import polars as pl
import financedatabase as fd
import importlib
import sys

script_dir = os.getcwd()
root_dir = os.path.abspath(os.path.join(script_dir, ".."))
if root_dir not in sys.path:
    sys.path.insert(0, root_dir)

import config # noqa: E402 # type: ignore

importlib.reload(config)

print(f"System Mode: {config.SYSTEM_MODE}")
data_files = glob.glob("../data/*.csv")
downloaded_tickers = [os.path.splitext(os.path.basename(f))[0] for f in data_files]

downloaded_tickers = [t for t in downloaded_tickers if "_5m" in t and t != "SPY_5m"]
    
print(f"Found {len(downloaded_tickers)} downloaded tickers for analysis: {downloaded_tickers}")

raw_master = pl.read_csv("../data/ticker_master.csv")

equities = fd.Equities()
meta_mapping = pl.from_pandas(equities.select(), include_index=True).select([
    pl.col("symbol").alias("ticker"),
    pl.col("sector"),
    pl.col("industry")
])

master_df = raw_master.join(meta_mapping, on="ticker", how="left")

# Custom Industry Overrides
CUSTOM_OVERRIDES = {
    "Auto Parts Retail": ["AZO", "ORLY"],
    "Aerospace and Defense": ["LMT", "NOC", "GD", "RTX"],
    "Pharmaceutical Retail": ["CVS", "WBA"],
    "Healthcare": ["MDT", "SYK"]
}

expr = pl.col("industry") 

for industry_name, tickers in CUSTOM_OVERRIDES.items():
    expr = pl.when(pl.col("ticker").is_in(tickers)).then(pl.lit(industry_name)).otherwise(expr)

master_df = master_df.with_columns(
    expr.alias("industry")
)

System Mode: P
Found 96 downloaded tickers for analysis: ['AAL_5m', 'ABBV_5m', 'AEP_5m', 'AMAT_5m', 'AMD_5m', 'AMGN_5m', 'AMT_5m', 'AON_5m', 'APD_5m', 'AVGO_5m', 'AZO_5m', 'BAC_5m', 'BJ_5m', 'BMY_5m', 'BX_5m', 'CAT_5m', 'CCL_5m', 'CDNS_5m', 'CME_5m', 'COF_5m', 'COP_5m', 'COST_5m', 'CRUS_5m', 'CSX_5m', 'CVS_5m', 'CVX_5m', 'C_5m', 'DAL_5m', 'DE_5m', 'DG_5m', 'DHI_5m', 'DHR_5m', 'DLTR_5m', 'DUK_5m', 'EOG_5m', 'EXC_5m', 'EXR_5m', 'FCX_5m', 'FDX_5m', 'GD_5m', 'GS_5m', 'HAL_5m', 'HD_5m', 'ICE_5m', 'JPM_5m', 'KKR_5m', 'KO_5m', 'LEN_5m', 'LIN_5m', 'LMT_5m', 'LOW_5m', 'LRCX_5m', 'MA_5m', 'MCO_5m', 'MDT_5m', 'MPC_5m', 'MSFT_5m', 'MS_5m', 'NEM_5m', 'NOC_5m', 'NSC_5m', 'NUE_5m', 'NVDA_5m', 'ORCL_5m', 'ORLY_5m', 'PAYX_5m', 'PEP_5m', 'PFE_5m', 'PNC_5m', 'PSA_5m', 'RCL_5m', 'RSG_5m', 'RTX_5m', 'SBAC_5m', 'SCCO_5m', 'SLB_5m', 'SNPS_5m', 'SO_5m', 'SPGI_5m', 'STLD_5m', 'SYK_5m', 'TGT_5m', 'TMO_5m', 'TOL_5m', 'T_5m', 'UAL_5m', 'UPS_5m', 'USB_5m', 'VLO_5m', 'VZ_5m', 'V_5m', 'WBA_5m', 'WFC_5m', 'WMT_5m', '

In [2]:
# CELL 2

import itertools
import numpy as np
from datetime import datetime, timezone
from statsmodels.tsa.vector_ar.vecm import coint_johansen

def test_johansen_pair(df, ticker_a, ticker_b):
    log_p1 = np.log(df[ticker_a].to_numpy())
    log_p2 = np.log(df[ticker_b].to_numpy())
    endog = np.column_stack([log_p1, log_p2])
    
    res = coint_johansen(endog, det_order=0, k_ar_diff=1)
    
    trace_stat_r0 = res.lr1[0]
    
    trace_crit_90_r0 = res.cvt[0, 0]
    
    is_cointegrated = (trace_stat_r0 > trace_crit_90_r0)
    
    weights = res.evec[:, 0]
    hedge_ratio = -weights[1] / weights[0]
    
    return is_cointegrated, trace_stat_r0, trace_crit_90_r0, hedge_ratio

def scan_universe_by_industry(metadata_df, data_folder="../data", file_suffix="_5m"):
    valid_pairs = []
    
    downloaded_files = []
    for f in glob.glob(f"{data_folder}/*{file_suffix}.csv"):
        base_name = os.path.splitext(os.path.basename(f))[0]
        pure_ticker = base_name.replace(file_suffix, "")
        downloaded_files.append(pure_ticker)
    
    local_metadata = metadata_df.filter(pl.col("ticker").is_in(downloaded_files))
    unique_industries = local_metadata["industry"].unique().drop_nulls().to_list()
    print(f"Industries found: {unique_industries}\n")
    print("="*50)
    
    start_dt = datetime.strptime(config.SELECTION_START_DATE, "%Y-%m-%d").replace(tzinfo=timezone.utc)
    end_dt = datetime.strptime(config.SELECTION_END_DATE, "%Y-%m-%d").replace(tzinfo=timezone.utc)
    
    for industry in unique_industries:
        industry_tickers = sorted(list(set(local_metadata.filter(pl.col("industry") == industry)["ticker"].to_list())))
        
        if len(industry_tickers) < 2:
            continue
            
        print(f"Scanning {industry} | Tickers to cross-test: {industry_tickers}")
        pairs_list = list(itertools.combinations(industry_tickers, 2))
        
        for ticker_a, ticker_b in pairs_list:
            try:
                file_a = f"{data_folder}/{ticker_a}{file_suffix}.csv"
                file_b = f"{data_folder}/{ticker_b}{file_suffix}.csv"
                
                df_a = pl.read_csv(file_a, try_parse_dates=True).select(["date", "close"]).rename({"close": ticker_a})
                df_b = pl.read_csv(file_b, try_parse_dates=True).select(["date", "close"]).rename({"close": ticker_b})
                
                pair_df = df_a.join(df_b, on="date", how="inner").sort("date").drop_nulls()

                pair_df = pair_df.filter(
                    (pl.col("date") >= start_dt) & 
                    (pl.col("date") <= end_dt)
                )
                
                if len(pair_df) < 4000:
                    print(f"  [SKIP] {ticker_a} & {ticker_b} | Insufficient overlapping data ({len(pair_df)} bars)")
                    continue
                    
                is_coint, t_stat, c_val, hedge_ratio = test_johansen_pair(pair_df, ticker_a, ticker_b)
                
                if is_coint:
                    print(f"  [PASS] {ticker_a} & {ticker_b} | Trace: {t_stat:.2f} > {c_val:.2f}")
                    valid_pairs.append({
                        "ticker_a": ticker_a, "ticker_b": ticker_b, "industry": industry,
                        "trace_stat": t_stat, "initial_johansen_hedge_ratio": hedge_ratio
                    })
                else:
                    print(f"  [FAIL] {ticker_a} & {ticker_b} | Trace: {t_stat:.2f} < {c_val:.2f}")
                    
            except Exception as e:
                print(f"  [ERROR] Failed to process {ticker_a} & {ticker_b}: {e}") 
                continue
                
        print("-" * 50)
                
    return pl.DataFrame(valid_pairs)

print(f"System Mode: {config.SYSTEM_MODE}")
cointegrated_pairs_df = scan_universe_by_industry(master_df)

System Mode: P
Industries found: ['Household Products', 'Machinery', 'Software', 'Banks', 'Semiconductors & Semiconductor Equipment', 'Metals & Mining', 'Road & Rail', 'Electric Utilities', 'Capital Markets', 'Consumer Finance', 'Oil, Gas & Consumable Fuels', 'Professional Services', 'Household Durables', 'Transportation Infrastructure', 'Biotechnology', 'Hotels, Restaurants & Leisure', 'Construction & Engineering', 'Pharmaceuticals', 'Healthcare', 'Air Freight & Logistics', 'Aerospace and Defense', 'Chemicals', 'Airlines', 'Auto Parts Retail', 'Pharmaceutical Retail', 'Insurance', 'Diversified Telecommunication Services', 'Beverages', 'Multi-Utilities', 'Equity Real Estate Investment Trusts (REITs)', 'Energy Equipment & Services']

Scanning Household Products | Tickers to cross-test: ['BJ', 'COST', 'DG', 'DLTR', 'TGT', 'WMT']
  [PASS] BJ & COST | Trace: 19.54 > 13.43
  [PASS] BJ & DG | Trace: 14.26 > 13.43
  [PASS] BJ & DLTR | Trace: 13.44 > 13.43
  [PASS] BJ & TGT | Trace: 19.56 > 13

In [3]:
# CELL 3

import networkx as nx

def maximize_independent_pairs(df):
    G = nx.Graph()
    
    for row in df.iter_rows(named=True):
        G.add_edge(
            row["ticker_a"], 
            row["ticker_b"], 
            weight=row["trace_stat"], 
            full_row=row
        )
        
    optimal_matching = nx.max_weight_matching(G, maxcardinality=True)
    
    optimal_rows = []
    for ticker_a, ticker_b in optimal_matching:
        edge_data = G.edges[ticker_a, ticker_b]["full_row"]
        optimal_rows.append(edge_data)
        
    final_df = pl.DataFrame(optimal_rows)
    
    print(f"Reduced {len(df)} overlapping pairs into {len(final_df)} independent pairs.")
    return final_df

print(f"System Mode: {config.SYSTEM_MODE}")
final_pairs_df = maximize_independent_pairs(cointegrated_pairs_df)

System Mode: P
Reduced 47 overlapping pairs into 25 independent pairs.


In [4]:
# CELL 4

print(f"System Mode: {config.SYSTEM_MODE}")
print(f"Saved to {config.PAIRS_OUTPUT_FILE}")
final_pairs_df.write_csv(f"../outputs/{config.PAIRS_OUTPUT_FILE}")

System Mode: P
Saved to active_johansen_pairs.csv
